# 04 · Long-Term Memory: remembering across conversations

Short-term memory (the message list) disappears when the program stops. A customer who returns tomorrow is a stranger again.

**Long-term memory** saves the important facts **outside the conversation**, in a file or a database, so a brand new session can load them.

The idea, in three steps:
1. During a chat, the customer shares things (name, preferences, past issues).
2. When the chat ends, the model **extracts the lasting facts** (structured output from Module 02) and we **save** them.
3. When the customer returns, we **load** the facts into the system prompt.

We will save to a simple JSON file. A real company would use a database.

In [1]:
import os
import json
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()   # read your keys from the .env file
assert os.getenv("OPENAI_API_KEY"), "Add OPENAI_API_KEY to your .env file (see .env.example)"

client = OpenAI()        # finds OPENAI_API_KEY on its own
MODEL = "gpt-4o-mini"    # a small, cheap model is plenty for this module
print("Ready")

Ready


## Saving and loading
Memory is a dictionary: customer id ➜ list of facts. We keep it in `customer_memory.json`.

In [2]:
from pathlib import Path

MEMORY_FILE = Path("customer_memory.json")
MEMORY_FILE.unlink(missing_ok=True)      # start this demo with a clean slate

def load_memory(customer_id):
    """Return the list of facts we remember about a customer (empty if none)."""
    if not MEMORY_FILE.exists():
        return []
    return json.loads(MEMORY_FILE.read_text()).get(customer_id, [])

def save_memory(customer_id, facts):
    if MEMORY_FILE.exists():
        data = json.loads(MEMORY_FILE.read_text())    # load what is already saved
    else:
        data = {}                                     # nothing saved yet
    data[customer_id] = facts
    MEMORY_FILE.write_text(json.dumps(data, indent=2))

def forget_customer(customer_id):
    """Customers can ask us to delete what we remember about them."""
    data = json.loads(MEMORY_FILE.read_text())
    data.pop(customer_id, None)
    MEMORY_FILE.write_text(json.dumps(data, indent=2))

## Extracting the facts
We do not save the whole conversation. We ask the model for the few **lasting facts** worth remembering, as structured output.
The prompt also says what **not** to store. Memories are personal data, so only keep what helps and never keep payment details.

In [3]:
from pydantic import BaseModel

class MemoryNotes(BaseModel):
    facts: list[str]     # short facts, for example "Prefers email over phone"

def extract_facts(conversation_text, old_facts):
    """Ask the model for the updated list of lasting facts about this customer."""
    response = client.chat.completions.parse(
        model=MODEL,
        messages=[
            {"role": "system", "content": (
                "You keep notes about a customer for a support team. "
                "Given the existing notes and the latest conversation, return the updated list of at most 5 short facts. "
                "Keep only lasting facts: name, preferences, past issues and their outcome. "
                "Remove duplicates. Never store payment details, passwords or addresses."
            )},
            {"role": "user", "content": f"Existing notes: {old_facts}\n\nConversation:\n{conversation_text}"},
        ],
        response_format=MemoryNotes,
    )
    return response.choices[0].message.parsed.facts

## Session 1: a first conversation
The customer chats, then the session ends and we save the facts.

In [4]:
CUSTOMER = "priya@example.com"

def start_session(customer_id):
    """Start a conversation. Anything we remember about this customer goes into the system prompt."""
    system = "You are a friendly support assistant for an online shop. Keep answers to 2 sentences."
    facts = load_memory(customer_id)
    if facts:
        system += "\n\nWhat you remember about this customer from earlier chats:\n- " + "\n- ".join(facts)
    return [{"role": "system", "content": system}]

def chat(history, user_text):
    history.append({"role": "user", "content": user_text})
    response = client.chat.completions.create(model=MODEL, messages=history)
    reply = response.choices[0].message.content
    history.append({"role": "assistant", "content": reply})
    print(f"Customer : {user_text}\nAssistant: {reply}\n")

def end_session(customer_id, history):
    """Extract the lasting facts from the conversation and save them."""
    text = "\n".join(f"{m['role']}: {m['content']}" for m in history[1:])    # skip the system message
    facts = extract_facts(text, load_memory(customer_id))
    save_memory(customer_id, facts)
    return facts

history = start_session(CUSTOMER)
chat(history, "Hi, I'm Priya, but please call me Pri.")
chat(history, "My headphones from order A1001 were refunded last month, so I'm hoping the replacement goes smoothly.")
chat(history, "Please always contact me by email, never by phone.")

print("Saved facts:", end_session(CUSTOMER, history))

Customer : Hi, I'm Priya, but please call me Pri.
Assistant: Hi Pri! How can I assist you today?



Customer : My headphones from order A1001 were refunded last month, so I'm hoping the replacement goes smoothly.
Assistant: I'm glad to hear your refund was processed! If you have any concerns about the replacement order, just let me know and I'll be happy to help.



Customer : Please always contact me by email, never by phone.
Assistant: Got it, Pri! We'll make sure to communicate with you via email only.



Saved facts: ['Name: Priya (preferred name: Pri)', 'Last order: Headphones (order A1001)', 'Refund issued for order A1001', 'Preference for communication: Email only']


The conversation is over and the message list is gone, but the facts were saved to disk:

In [5]:
print(MEMORY_FILE.read_text())

{
  "priya@example.com": [
    "Name: Priya (preferred name: Pri)",
    "Last order: Headphones (order A1001)",
    "Refund issued for order A1001",
    "Preference for communication: Email only"
  ]
}


## Session 2: the customer comes back
This is a **brand new** message list. The only thing carried over is the saved file.

In [6]:
history = start_session(CUSTOMER)
chat(history, "Hi again! How will you contact me about my replacement?")

Customer : Hi again! How will you contact me about my replacement?
Assistant: Hi Pri! We will contact you via email regarding your replacement, so please keep an eye on your inbox for updates.



The assistant knows the name, the preference and the history, **without being told again**.

## Memory belongs to one customer
Another customer gets a fresh start, so one person's memories never leak into another's chat.

In [7]:
history = start_session("rahul@example.com")
chat(history, "Hi, do you remember me? How will you contact me?")

Customer : Hi, do you remember me? How will you contact me?
Assistant: I'm glad to assist you, but I don't have the ability to remember past conversations. You can always reach out to us through our website or customer support email for assistance!



## The right to be forgotten
Because we store personal facts, customers must be able to remove them.

In [8]:
forget_customer(CUSTOMER)
print("Facts after forgetting:", load_memory(CUSTOMER))

MEMORY_FILE.unlink(missing_ok=True)      # tidy up the demo file

Facts after forgetting: []


## Recap
* **Short-term memory** = the message list. **Long-term memory** = facts saved outside the chat.
* Extract only the **lasting facts**, using structured output, instead of saving whole conversations.
* Load them into the system prompt at the start of a new session.
* Keep memory **per customer**, store **as little as possible**, and let people **delete it**.

Next: give the agent from Module 02 a **knowledge base search tool**, so it decides when to look things up.